In [1]:
from typing import Dict, List, Callable, Any

## 工具类

In [2]:
class Tool:
    """
    工具属性说明：每个工具包括五部分
    name：唯一工具识别编号
    description：工具用途，便于LLM调用
    input_schema：工具期待的数据格式
    output_shcema：工具返回的格式、
    func：具体工作的函数
    """
    def __init__(
            self,
            name: str,
            description: str,
            input_schema: Dict[str, Any],
            output_schema: Dict[str, Any],
            func: Callable[..., Any],
    ):
        self.name = name
        self.description = description
        self.input_schema = input_schema
        self.output_schema = output_schema
        self.func = func

    def __call__(self, **kwargs):
        return self.func(**kwargs)

## 工具注册
工具注册类用来记录工具及其用法，并随时进行管理和调用

In [26]:
from typing import Union, Literal
from pydantic import BaseModel

class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
    
    def register(self, tool: Tool):
        self.tools[tool.name] = tool
    
    def get(self, name: str) -> Tool:
        if name not in self.tools.keys():
            raise ValueError(f"Tool '{name}' not found")
        return self.tools[name]
    
    def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema.model_json_schema()
            } 
            for tool in self.tools.values()
        ]
    
    def get_tool_call_args_type(self) -> Union[BaseModel]:
        input_args_models = [tool.input_schema for tool in self.tools.values()]
        tool_call_args = Union[tuple(input_args_models)]
        return tool_call_args
    
    def get_tool_names(self) -> Literal[None]:
        return Literal[*self.tools.keys()]

## list_tools()函数告诉LLM它可以做什么
它返回如下格式的描述

In [27]:
[
    {
        "name": "add",
        "description": "Add two Number",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"}
            },
            "required": ["a", "b"]
        }
    }
]

[{'name': 'add',
  'description': 'Add two Number',
  'input_schema': {'type': 'object',
   'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
   'required': ['a', 'b']}}]

## get_tool_names()
可以阻止大模型幻觉，其将所有的工具列表返回，强制让大模型选择

## 注册工具

In [28]:
def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b

In [29]:
class ToolAddArgs(BaseModel):
    a: int
    b: int

class ToolMultiplyArgs(BaseModel):
    a: int
    b: int

In [30]:
registry = ToolRegistry()

add_args = {
    "name": "add",
    "description": "Add two numbers",
    "input_schema": ToolAddArgs,
    "output_schema": {"result": "int"},
    "func": add
}

mul_args = {
    "name": "mul",
    "description": "Multiply two numbers",
    "input_schema": ToolMultiplyArgs,
    "output_schema": {"result": "int"},
    "func": multiply
}

registry.register(
    Tool(**add_args)
)

registry.register(
    Tool(**mul_args)
)

In [31]:
registry.get_tool_names()

typing.Literal['add', 'mul']

In [32]:
registry.list_tools()

[{'name': 'add',
  'description': 'Add two numbers',
  'input_schema': {'properties': {'a': {'title': 'A', 'type': 'integer'},
    'b': {'title': 'B', 'type': 'integer'}},
   'required': ['a', 'b'],
   'title': 'ToolAddArgs',
   'type': 'object'}},
 {'name': 'mul',
  'description': 'Multiply two numbers',
  'input_schema': {'properties': {'a': {'title': 'A', 'type': 'integer'},
    'b': {'title': 'B', 'type': 'integer'}},
   'required': ['a', 'b'],
   'title': 'ToolMultiplyArgs',
   'type': 'object'}}]

In [33]:
registry.get('add')

## Pydantic类型安全
使用Pydantic而不是JSON是因为类型安全考虑。当使用LLM时，最大的挑战是保证返回的数据格式可以被代码信任执行。即使今天大模型已经被广泛训练如何使用工具，但幻觉问题依然存在，这就是为什么我们需要保证类型安全。
Pydantic模型像合约一样运行：
1.自动验证输入数据
2.数据不合法时提供纠错信息
3.允许IDE自动补全
4.生成现代LLM可以使用的JSON schemas数据格式

In [ ]:
ToolNameLiteral = registry.get_tool_names()
ToolArgsUnion = registry.get_tool_call_args_type()

class ToolCall(BaseModel):
    action: Literal["tool"]
    thought: str
    tool_name: ToolNameLiteral
    args: ToolArgsUnion

class FinalAnswer(BaseModel):
    action: Literal["final"]
    answer: str
LLMResponse = Union[ToolCall, FinalAnswer]

ReAct模式下，LLM必须选择一种行动类型（tool或者final）

https://pub.towardsai.net/creating-an-advanced-ai-agent-from-scratch-with-python-in-2025-part-1-ce74a23f6514

## LLM Wrapper
现在可以整合谷歌Gemini API。这里可以自行替换不同的LLM厂商，付费免费皆可。如果后续私有部署在集群上，可选择将模型参数放上去。

In [ ]:
import json
